### Connexion à la DB DuckDB

In [1]:
import duckdb
import os
from pathlib import Path
from typing import List
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Connexion à la DB / Import des Data


In [2]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))

In [3]:
# 2. Query to list all tables in the database
# DuckDB specific way to list tables
tables_info = con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

print(f"Found {len(tables_info)} tables in the database:\n")

if len(tables_info) > 0:
    for i, table_name in enumerate(tables_info['table_name']):
        print(f"{i+1}. {table_name}")
else:
    print("No tables found in the database.")

Found 3 tables in the database:

1. all_events
2. loaded_files
3. user_events


In [4]:
all_events_count = con.execute("SELECT COUNT(*) FROM all_events ").fetchone()[0]

print(f"Taille de la table all_events : {all_events_count} logs")

all_events_df = con.execute("SELECT * FROM all_events ORDER BY RANDOM() LIMIT 5000").fetch_df()
print(all_events_df)

Taille de la table all_events : 288779227 logs


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

              event_time event_type product_id          category_id  \
0    2019-12-31 18:17:35       view    1004362  2232732093077520756   
1    2019-12-07 10:45:45       view   17800272  2232732086257582287   
2    2019-10-04 13:16:22       view    1701343  2053013553031414015   
3    2019-12-07 08:11:25       view    1004840  2232732093077520756   
4    2020-02-01 05:23:26       view    1004785  2232732093077520756   
...                  ...        ...        ...                  ...   
4995 2019-12-21 16:19:35       view    1002662  2053013555631882655   
4996 2019-11-01 11:59:02       view   22700128  2053013556168753601   
4997 2019-10-22 03:16:25       view   16400119  2053013558249128509   
4998 2019-12-23 13:40:50       view   21400808  2232732082063278200   
4999 2019-12-05 14:47:08       view    3601448  2232732092297380188   

                      category_code    brand    price    user_id  \
0          construction.tools.light    apple  1155.50  595353709   
1          

In [5]:
all_events_df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-12-31 18:17:35,view,1004362,2232732093077520756,construction.tools.light,apple,1155.50,595353709,df3bb5cf-fe71-4177-9211-c46f2f7cc657
1,2019-12-07 10:45:45,view,17800272,2232732086257582287,electronics.tablet,zeta,95.09,553237456,d1d54dbd-3b58-4e4c-bda9-b649e7d892b2
2,2019-10-04 13:16:22,view,1701343,2053013553031414015,computers.peripherals.monitor,samsung,386.08,514703286,7de56fca-9bb8-4eb9-a6c7-65a2be5de70d
3,2019-12-07 08:11:25,view,1004840,2232732093077520756,construction.tools.light,huawei,900.15,523060195,544deedd-17e1-4203-97e6-c4883ea1d96c
4,2020-02-01 05:23:26,view,1004785,2232732093077520756,construction.tools.light,huawei,230.25,608872640,b3fbd9bf-ec30-4c64-9f42-12f6f86a567f
...,...,...,...,...,...,...,...,...,...
4995,2019-12-21 16:19:35,view,1002662,2053013555631882655,electronics.smartphone,xiaomi,203.09,590391336,890d88c0-068d-4a65-8eee-ebc3c4bfd221
4996,2019-11-01 11:59:02,view,22700128,2053013556168753601,None,stels,68.21,514248623,a9c86ac7-5a21-4af2-b41a-56d31e5ec263
4997,2019-10-22 03:16:25,view,16400119,2053013558249128509,None,rondell,110.10,546770115,956a31a3-26d7-4228-b273-b9932fbc6fab
4998,2019-12-23 13:40:50,view,21400808,2232732082063278200,electronics.clocks,casio,155.22,556899764,c626410e-259e-4e4d-b6c8-e0c5dc6e1603


In [6]:
# STEP 0 — Raw Event Data (Input)
# Assuming `all_events_df` contains the raw data
raw_data = all_events_df
# Drop rows where 'category_code' is None or NaN
all_events_df = all_events_df.dropna(subset=["category_code"])

# Verify the result
print(all_events_df)

              event_time event_type product_id          category_id  \
0    2019-12-31 18:17:35       view    1004362  2232732093077520756   
1    2019-12-07 10:45:45       view   17800272  2232732086257582287   
2    2019-10-04 13:16:22       view    1701343  2053013553031414015   
3    2019-12-07 08:11:25       view    1004840  2232732093077520756   
4    2020-02-01 05:23:26       view    1004785  2232732093077520756   
...                  ...        ...        ...                  ...   
4993 2020-01-10 05:28:53       view    1005112  2232732093077520756   
4994 2019-12-14 13:36:12       view    1004754  2232732093077520756   
4995 2019-12-21 16:19:35       view    1002662  2053013555631882655   
4998 2019-12-23 13:40:50       view   21400808  2232732082063278200   
4999 2019-12-05 14:47:08       view    3601448  2232732092297380188   

                      category_code    brand    price    user_id  \
0          construction.tools.light    apple  1155.50  595353709   
1          

In [7]:
# STEP 1 — User → List of Purchased Categories
user_categories = all_events_df.groupby("user_id")["category_code"].apply(list)
print(user_categories)

user_id
392716918             [furniture.bedroom.bed]
436354000    [construction.components.faucet]
445019763     [appliances.environment.vacuum]
465083278          [construction.tools.light]
468888711                [computers.notebook]
                           ...               
621346564                   [accessories.bag]
621587121              [electronics.video.tv]
621751044          [construction.tools.light]
621788333          [construction.tools.light]
621950672                         [kids.toys]
Name: category_code, Length: 4093, dtype: object


In [8]:
# STEP 2 — Hierarchy Expansion
def expand_hierarchy(categories):
    expanded = []
    for category in categories:
        parts = category.split(".")
        expanded.extend([".".join(parts[:i+1]) for i in range(len(parts))])
    return expanded

user_hierarchy = user_categories.apply(expand_hierarchy)
print(user_hierarchy)



user_id
392716918    [furniture, furniture.bedroom, furniture.bedro...
436354000    [construction, construction.components, constr...
445019763    [appliances, appliances.environment, appliance...
465083278    [construction, construction.tools, constructio...
468888711                      [computers, computers.notebook]
                                   ...                        
621346564                       [accessories, accessories.bag]
621587121    [electronics, electronics.video, electronics.v...
621751044    [construction, construction.tools, constructio...
621788333    [construction, construction.tools, constructio...
621950672                                    [kids, kids.toys]
Name: category_code, Length: 4093, dtype: object


In [9]:
# STEP 3 — User as “Document” of Tokens
user_tokens = user_hierarchy.apply(lambda x: " ".join(x))
print(user_tokens)

user_id
392716918    furniture furniture.bedroom furniture.bedroom.bed
436354000    construction construction.components construct...
445019763    appliances appliances.environment appliances.e...
465083278    construction construction.tools construction.t...
468888711                         computers computers.notebook
                                   ...                        
621346564                          accessories accessories.bag
621587121    electronics electronics.video electronics.vide...
621751044    construction construction.tools construction.t...
621788333    construction construction.tools construction.t...
621950672                                       kids kids.toys
Name: category_code, Length: 4093, dtype: object


In [10]:
# STEP 4 — TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(user_tokens)
print(tfidf_matrix.toarray())
print(vectorizer.get_feature_names_out())



[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
['accessories' 'acoustic' 'air_conditioner' 'air_heater' 'alarm' 'apparel'
 'appliances' 'audio' 'auto' 'bag' 'ballet_shoes' 'bath' 'bathroom' 'bed'
 'bedroom' 'belt' 'bicycle' 'blanket' 'blender' 'cabinet' 'camera'
 'carriage' 'cartrige' 'chair' 'climate' 'clocks' 'coffee_grinder'
 'coffee_machine' 'components' 'compressor' 'computers' 'construction'
 'cooler' 'costume' 'country_yard' 'cpu' 'cultivator' 'desktop' 'diapers'
 'dishwasher' 'diving' 'dolls' 'dress' 'drill' 'ebooks' 'electronics'
 'environment' 'espadrilles' 'faucet' 'fmcg' 'fryer' 'furniture'
 'generator' 'glove' 'grill' 'hair_cutter' 'hammok' 'hdd' 'headphone'
 'hob' 'hood' 'iron' 'ironing_board' 'jeans' 'juicer' 'jumper' 'keds'
 'kettle' 'keyboard' 'kids' 'kitchen' 'lawn_mower' 'light' 'living_room'
 'massager' 'meat_grinder' 'medicine' 'memory' 'microphone' 'microwa

In [11]:
# STEP 5 — Dimensionality Reduction (SVD)
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=10, random_state=42)
reduced_matrix = svd.fit_transform(tfidf_matrix)
print(reduced_matrix)



[[ 8.85085318e-05  1.03478131e-05  1.50837101e-02 ... -1.11681435e-03
  -1.99855023e-01 -4.94655584e-04]
 [ 4.52830922e-01 -1.09626582e-03 -4.97966965e-04 ...  5.17721491e-05
   4.19530684e-05 -3.84674719e-05]
 [ 5.90488507e-04  6.19521115e-04  6.24881777e-01 ... -2.26872041e-03
  -4.64682487e-01 -1.05364694e-03]
 ...
 [ 9.99457355e-01 -2.39442479e-03 -1.05478686e-03 ...  1.25670783e-04
  -1.23517882e-04  5.35582237e-05]
 [ 9.99457355e-01 -2.39442479e-03 -1.05478686e-03 ...  1.25670783e-04
  -1.23517882e-04  5.35582237e-05]
 [ 3.60711465e-06 -1.02175629e-08 -1.60991289e-08 ... -9.93020119e-07
   3.45469884e-07 -4.15167242e-06]]


In [12]:
# STEP 6 — Final Clustering Input
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(reduced_matrix)
print(kmeans.labels_)

# Optional: Alternative Path (Simpler, No TF-IDF)
# Level-1 Only (Coarse)
def level_1_only(categories):
    return [cat.split(".")[0] for cat in categories]

user_level_1 = user_categories.apply(level_1_only)
user_level_1_counts = user_level_1.apply(lambda x: pd.Series(x).value_counts(normalize=True)).fillna(0)
print(user_level_1_counts)

# Interpretation of Clusters
centroids = kmeans.cluster_centers_
print("Cluster centroids:", centroids)

[1 3 0 ... 3 3 1]
           furniture  construction  appliances  computers  kids  apparel  \
user_id                                                                    
392716918        1.0           0.0         0.0        0.0   0.0      0.0   
436354000        0.0           1.0         0.0        0.0   0.0      0.0   
445019763        0.0           0.0         1.0        0.0   0.0      0.0   
465083278        0.0           1.0         0.0        0.0   0.0      0.0   
468888711        0.0           0.0         0.0        1.0   0.0      0.0   
...              ...           ...         ...        ...   ...      ...   
621346564        0.0           0.0         0.0        0.0   0.0      0.0   
621587121        0.0           0.0         0.0        0.0   0.0      0.0   
621751044        0.0           1.0         0.0        0.0   0.0      0.0   
621788333        0.0           1.0         0.0        0.0   0.0      0.0   
621950672        0.0           0.0         0.0        0.0   1.0      0